# tealql playground

Edit the TEAL in **Cell 1**, then re-run **Cell 2** (rewrites the source) and **Cell 3** (runs analyses). Everything is reconstructed in-process from the `.teal` source — pure Python, no database, no build step.

In [ ]:
# --- 1. TEAL source ---
# Edit me, then re-run cells 2 and 3.

TEAL_SOURCE = r"""
#pragma version 10
// Sample: external arg flowing into a box write, no validation.

pushbytes "k"
txna ApplicationArgs 0
box_put

pushint 1
return
"""


In [ ]:
# --- 2. Write the TEAL source to a temp dir ---
import tempfile
from pathlib import Path

PLAYGROUND = Path(tempfile.gettempdir()) / "tealql-playground"
SRC = PLAYGROUND / "src"
SRC.mkdir(parents=True, exist_ok=True)
(SRC / "prog.teal").write_text(TEAL_SOURCE.lstrip())
print(f"wrote {SRC / 'prog.teal'}")

In [ ]:
# --- 3. Run analyses against the source above ---
import sys, importlib
REPO_ROOT = Path.cwd().parent          # playground/ -> repo root
sys.path.insert(0, str(REPO_ROOT / "src"))

# Force-reload tealql if you've edited its source between runs
for mod in list(sys.modules):
    if mod == "tealql" or mod.startswith("tealql."):
        del sys.modules[mod]

from tealql.tealtools import SSAProgram
from tealql.tealtools.path_predicates import PathPredicateAnalysis
from tealql.tealtools.group_reasoning import analyze as group_shape
from tealql.tealtools.cost_analysis import render as render_cost
from tealql.tealtools.auth_domination import AuthDominationDetector
from tealql.security import NonUniqueBoxKeyDetector
from tealql.tealtools.dataflow.box import (
    detect_into_box_flows,
    detect_out_of_box_flows,
    detect_correlated_flows,
)
from tealql.tealtools.inner_txn_report import InnerTxnReport

prog = SSAProgram(str(SRC))
print(f"SSA: {len(prog.assignments)} assignments, "
      f"{len(prog.blocks)} BBs, {len(prog.phis)} phis\n")

# --- pick the analyses you care about ---

print("=== Path Predicates ===")
print(PathPredicateAnalysis(prog).render())
print()

print("=== Group Shape ===")
print(group_shape(prog).render())
print()

print("=== Cost ===")
print(render_cost(prog))
print()

print("=== Auth Domination ===")
vs = AuthDominationDetector(prog).detect()
print("\n".join(v.pretty() for v in vs) if vs else "(no violations)")
print()

print("=== Box DF: into-box ===")
vs = detect_into_box_flows(prog)
print("\n".join(v.pretty() for v in vs) if vs else "(no violations)")
print()

print("=== Box DF: out-of-box ===")
vs = detect_out_of_box_flows(prog)
print("\n".join(v.pretty() for v in vs) if vs else "(no violations)")
print()

print("=== Inner-txn Report ===")
print(InnerTxnReport(prog).render())
